# 01 · Python Fundamentals — a working notebook

Companion to **The Workstation Atlas**. The atlas maps your machine; this notebook drills the
language you drive it with.

**How to use it.** Each section is three cells: a *demo* you read and run, an *exercise* you fill
in, and a *check* that grades it. Checks print ✅ / ❌ / ⬜ — they never raise, so you can run the
whole notebook top to bottom at any stage of completion and nothing blows up.

Stubs start as `raise NotImplementedError`, which reports as ⬜ *not attempted*. Replace the
`raise` with real code and re-run the check cell.

> Kernel: `Python 3.14 (workstation-atlas)` — the project-local `.venv`, rung **S1** on the
> packaging ladder from atlas page 8.

In [ ]:
"""Run me first — defines the grader used by every check cell."""
from __future__ import annotations

_SCORE = {"pass": 0, "fail": 0, "todo": 0}


def check(label: str, thunk, want) -> bool:
    """Grade one exercise. `thunk` is a zero-arg callable; nothing here ever raises."""
    try:
        got = thunk()
    except NotImplementedError:
        _SCORE["todo"] += 1
        print(f"⬜ {label} — not attempted yet")
        return False
    except Exception as exc:                      # noqa: BLE001 — grading, not production
        _SCORE["fail"] += 1
        print(f"❌ {label} — raised {type(exc).__name__}: {exc}")
        return False
    if got == want:
        _SCORE["pass"] += 1
        print(f"✅ {label}")
        return True
    _SCORE["fail"] += 1
    print(f"❌ {label}\n     got:  {got!r}\n     want: {want!r}")
    return False


def scoreboard() -> None:
    total = sum(_SCORE.values())
    print(f"passed {_SCORE['pass']}/{total}   failed {_SCORE['fail']}   untouched {_SCORE['todo']}")


import sys
print(f"python {sys.version.split()[0]}  ·  {sys.prefix}")

---
## 1 · The four containers

`list` ordered + mutable · `tuple` ordered + immutable (so it can be a dict key) ·
`dict` mapping, insertion-ordered since 3.7 · `set` unordered, unique, O(1) membership.

Picking the right one is usually the whole optimisation. Membership testing in a `list` is O(n);
in a `set` it is O(1). That difference is the kind of thing atlas page 6 means by *measure before
you optimise* — except here you can reason it out without measuring.

In [ ]:
from timeit import timeit

haystack_list = list(range(50_000))
haystack_set = set(haystack_list)
needle = 49_999                       # worst case: last element

t_list = timeit(lambda: needle in haystack_list, number=200)
t_set = timeit(lambda: needle in haystack_set, number=200)
print(f"list membership: {t_list:.5f}s")
print(f"set  membership: {t_set:.5f}s   ({t_list / t_set:,.0f}x faster)")

# tuples are hashable, lists are not — this is why tuples make good dict keys
grid = {(0, 0): "origin", (1, 2): "somewhere"}
print(grid[(1, 2)])
try:
    {[0, 0]: "nope"}
except TypeError as exc:
    print(f"TypeError: {exc}")

In [ ]:
def tally(words: list[str]) -> dict[str, int]:
    """Count how many times each word appears, preserving first-seen order.

    tally(["a", "b", "a"]) -> {"a": 2, "b": 1}
    Do it without collections.Counter — the point is the dict, not the import.
    """
    raise NotImplementedError


def unique_preserving_order(items: list[int]) -> list[int]:
    """Drop duplicates but keep the original order. Use a set for the seen-check."""
    raise NotImplementedError

In [ ]:
check("tally counts and orders",
      lambda: tally(["shell", "git", "shell", "uv", "git", "shell"]),
      {"shell": 3, "git": 2, "uv": 1})
check("unique_preserving_order keeps first occurrence",
      lambda: unique_preserving_order([3, 1, 3, 2, 1, 4]),
      [3, 1, 2, 4])

---
## 2 · Comprehensions

A comprehension is `expression for item in iterable if condition`, and it exists in list, set,
dict and generator flavours. Prefer it to `append` in a loop when you are *building* a collection;
prefer the explicit loop when you are *doing* something on each pass.

The trap: a comprehension that spans three lines and two `if`s is not clever, it is a loop wearing
a disguise. Atlas page 11's quality funnel would flag it at the lint layer.

In [ ]:
paths = ["~/.bun/bin", "~/.local/bin", "/opt/homebrew/bin", "/usr/bin", "~/dev/flutter/bin"]

print([p for p in paths if p.startswith("~")])            # list
print({p.split("/")[-1] for p in paths})                  # set  — note "bin" collapses
print({p: len(p) for p in paths})                         # dict
print(sum(len(p) for p in paths))                         # generator — no list built

# nested: order of `for` clauses reads exactly like nested loops
print([(a, b) for a in "xy" for b in (1, 2)])

In [ ]:
def rung_names(path_entries: list[str]) -> list[str]:
    """Return the final path segment of every entry, lowercased, EXCLUDING any "bin".

    rung_names(["/opt/homebrew/bin", "~/dev/Flutter"]) -> ["flutter"]
    One comprehension. No loop, no append.
    """
    raise NotImplementedError


def squares_by_root(n: int) -> dict[int, int]:
    """{i: i*i} for i in 1..n inclusive, odd i only. One dict comprehension."""
    raise NotImplementedError

In [ ]:
check("rung_names filters and lowercases",
      lambda: rung_names(["/opt/homebrew/bin", "~/dev/Flutter", "~/.local/BIN", "/usr/Local"]),
      ["flutter", "local"])
check("squares_by_root is odd-only",
      lambda: squares_by_root(7),
      {1: 1, 3: 9, 5: 25, 7: 49})

---
## 3 · Generators and laziness

`yield` turns a function into a factory for a lazy stream. Nothing is computed until something
pulls. That is the same idea as a shell pipe on atlas page 3: `curl | python3 -m json.tool` does
not buffer the whole response before the second stage starts — each stage pulls from the one
before it.

Generators matter when the sequence is huge, infinite, or expensive per item.

In [ ]:
import sys
from itertools import islice


def counter(start: int = 0):
    """An infinite stream. Perfectly safe — nothing runs until you pull."""
    n = start
    while True:
        yield n
        n += 1


print(list(islice(counter(10), 5)))

eager = [x * x for x in range(100_000)]
lazy = (x * x for x in range(100_000))
print(f"list comprehension: {sys.getsizeof(eager):>8,} bytes")
print(f"generator expr:     {sys.getsizeof(lazy):>8,} bytes")

# generators are one-shot: exhausted is exhausted
g = (c for c in "abc")
print(list(g), list(g))

In [ ]:
from collections.abc import Iterator


def take_while_under(stream, limit: int) -> Iterator[int]:
    """Yield items from `stream` while they are strictly < limit, then stop.

    Must stay lazy: take_while_under(counter(0), 3) has to terminate even though
    the source is infinite. Use `yield`, and `return` (or break) to stop.
    """
    raise NotImplementedError


def running_total(numbers) -> Iterator[int]:
    """Yield the cumulative sum after each item: [1,2,3] -> 1, 3, 6."""
    raise NotImplementedError

In [ ]:
check("take_while_under terminates on an infinite source",
      lambda: list(take_while_under(counter(0), 4)),
      [0, 1, 2, 3])
check("running_total accumulates",
      lambda: list(running_total([1, 2, 3, 4])),
      [1, 3, 6, 10])
check("take_while_under really is lazy (not a materialised list)",
      lambda: isinstance(take_while_under(counter(0), 2), Iterator),
      True)

---
## 4 · Function signatures, `*args`, `**kwargs`, closures

Atlas page 4 opens with *how to invoke*: `$1 $@ $#` in the shell. Python's version is richer —
positional, keyword, variadic, keyword-only — and it has one famous trap.

**The mutable default.** A default argument is evaluated **once**, at definition time. A mutable
default is therefore shared across every call. This is the single most common Python bug that
survives code review.

In [ ]:
def broken(item, bucket=[]):          # noqa: B006 — deliberately wrong, for the demo
    bucket.append(item)
    return bucket


print(broken("a"), broken("b"), broken("c"))    # the bucket persists — surprise

def fixed(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket


print(fixed("a"), fixed("b"), fixed("c"))

def describe(required, *args, sep="·", **kwargs):
    return f"{required} | args={args} | sep={sep!r} | kwargs={kwargs}"


print(describe(1, 2, 3, sep="-", mode="fast"))

def make_prefixer(prefix: str):
    """A closure: the returned function captures `prefix` from the enclosing scope."""
    def prefixer(text: str) -> str:
        return f"{prefix}{text}"
    return prefixer


warn = make_prefixer("⚠ ")
print(warn("disk almost full"))

In [ ]:
def make_counter(start: int = 0):
    """Return a zero-arg function that yields start, start+1, start+2 ... on each call.

    Needs a closure over mutable state — `nonlocal` is the keyword you want.
    """
    raise NotImplementedError


def merge_settings(*layers: dict, **overrides) -> dict:
    """Merge dicts left to right (later wins), then apply keyword overrides last.

    merge_settings({"a": 1}, {"a": 2, "b": 3}, b=9) -> {"a": 2, "b": 9}
    """
    raise NotImplementedError

In [ ]:
def _drive_counter():
    c = make_counter(5)
    return [c(), c(), c()]


check("make_counter holds state across calls", _drive_counter, [5, 6, 7])
check("two counters are independent",
      lambda: (lambda a, b: [a(), b(), a()])(make_counter(0), make_counter(100)),
      [0, 100, 1])
check("merge_settings layers then overrides",
      lambda: merge_settings({"a": 1, "keep": True}, {"a": 2, "b": 3}, b=9),
      {"a": 2, "keep": True, "b": 9})

---
## 5 · Decorators

A decorator is a function that takes a function and returns a replacement. `@timed` is just
`slow = timed(slow)` with nicer spelling.

This is the mechanism behind `@property`, `@dataclass`, `@pytest.fixture`, and every web
framework's routing table. Always wrap with `functools.wraps` or you silently destroy the wrapped
function's `__name__` and docstring — which is exactly the kind of invisible damage atlas page 10
means by *writing for the absent reader*.

In [ ]:
import functools
import time


def timed(fn):
    @functools.wraps(fn)                     # ← keeps __name__/__doc__ intact
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        result = fn(*args, **kwargs)
        print(f"  {fn.__name__} took {time.perf_counter() - t0:.4f}s")
        return result
    return wrapper


@timed
def slow_sum(n: int) -> int:
    """Add up the first n integers, slowly."""
    return sum(range(n))


print(slow_sum(1_000_000))
print(f"__name__ survived: {slow_sum.__name__!r}")
print(f"__doc__  survived: {slow_sum.__doc__!r}")

In [ ]:
import functools


def memoize(fn):
    """Cache results by positional args so repeat calls are free.

    Keep __name__ and __doc__ intact (functools.wraps). Expose the cache dict as
    `wrapper.cache` so the check can inspect it. Assume args are hashable.
    """
    raise NotImplementedError


def repeat(times: int):
    """A decorator FACTORY: @repeat(3) makes the function run 3x and return a list of results.

    Three layers — repeat(times) -> decorator(fn) -> wrapper(*args).
    """
    raise NotImplementedError

In [ ]:
def _drive_memoize():
    calls = []

    @memoize
    def square(n):
        calls.append(n)
        return n * n

    square(4), square(4), square(5)
    return calls                              # only one entry per distinct arg


def _drive_repeat():
    @repeat(3)
    def ping():
        return "pong"
    return ping()


check("memoize actually caches", _drive_memoize, [4, 5])
check("memoize preserves metadata",
      lambda: memoize(lambda: None).__name__ == "<lambda>", True)
check("repeat(3) collects three results", _drive_repeat, ["pong", "pong", "pong"])

---
## 6 · Context managers

`with` guarantees cleanup, including when the body raises. That guarantee is the whole point —
`f.close()` after an exception never runs; `with open(...)` always does.

Two ways to write one: a class with `__enter__`/`__exit__`, or a generator with
`@contextlib.contextmanager` and a single `yield` wrapped in `try/finally`.

In [ ]:
import contextlib
import os


class Chdir:
    """Class flavour: enter returns the value bound by `as`, exit always runs."""

    def __init__(self, target):
        self.target, self.previous = target, None

    def __enter__(self):
        self.previous = os.getcwd()
        os.chdir(self.target)
        return self.target

    def __exit__(self, exc_type, exc, tb):
        os.chdir(self.previous)
        return False                          # False = do not swallow the exception


print("before:", os.getcwd())
with Chdir("/tmp") as where:
    print("inside:", os.getcwd(), f"(as {where})")
print("after :", os.getcwd())


@contextlib.contextmanager
def tag(name):
    """Generator flavour: everything before yield is __enter__, the finally block is __exit__."""
    print(f"<{name}>")
    try:
        yield name
    finally:
        print(f"</{name}>")


with tag("section"):
    print("  body")

with contextlib.suppress(ZeroDivisionError):  # cleanup still runs on the way out
    with tag("doomed"):
        1 / 0

In [ ]:
import contextlib


@contextlib.contextmanager
def collecting(sink: list):
    """Yield an `emit` callable; everything emitted lands in `sink`.

    On the way out — even if the body raised — append the string "closed" to sink.
    """
    raise NotImplementedError


class Suppressing:
    """A context manager that swallows exactly the exception types it was built with.

    Suppressing(ValueError) swallows ValueError, lets everything else propagate.
    You need all three: __init__ (store the types), __enter__ (return self),
    and __exit__ (return True to suppress, False to let it through).
    """

    def __init__(self, *exc_types):
        raise NotImplementedError

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        raise NotImplementedError

In [ ]:
import contextlib


def _drive_collecting_ok():
    sink = []
    with collecting(sink) as emit:
        emit("a"); emit("b")
    return sink


class Boom(Exception):
    """Deliberately NOT a RuntimeError — NotImplementedError subclasses RuntimeError,
    so suppressing RuntimeError here would swallow the stub's own signal and report
    a confusing ❌ instead of an honest ⬜."""


def _drive_collecting_raises():
    sink = []
    with contextlib.suppress(Boom):
        with collecting(sink) as emit:
            emit("a")
            raise Boom("boom")
    return sink                               # cleanup must still have run


def _drive_suppressing():
    with Suppressing(ValueError):
        raise ValueError("swallowed")
    try:
        with Suppressing(ValueError):
            raise TypeError("propagates")
    except TypeError:
        return "propagated"
    return "wrongly swallowed"


check("collecting gathers then closes", _drive_collecting_ok, ["a", "b", "closed"])
check("collecting closes even when the body raises", _drive_collecting_raises, ["a", "closed"])
check("Suppressing is selective", _drive_suppressing, "propagated")

---
## 7 · Dataclasses and typing

Type hints are not enforced at runtime — they are a contract read by humans and by the L1 tools on
atlas page 5: Ruff, Mypy, and your editor's language server. The hint is what lets go-to-definition
and rename actually work.

`@dataclass` writes `__init__`, `__repr__` and `__eq__` from the annotations. `frozen=True` makes
instances immutable and hashable.

In [ ]:
from dataclasses import dataclass, field, asdict


@dataclass(frozen=True, slots=True)
class Rung:
    """One rung on a ladder — e.g. S1 `uv venv` from atlas page 8."""

    level: str
    tool: str
    isolates: str = "nothing yet"
    tags: list[str] = field(default_factory=list)   # never `= []`, same trap as §4

    @property
    def label(self) -> str:
        return f"{self.level} · {self.tool}"


s1 = Rung("S1", "uv venv", "dependencies", ["python"])
print(s1)
print(s1.label)
print(asdict(s1))
print("equality is by value:", s1 == Rung("S1", "uv venv", "dependencies", ["python"]))

try:
    s1.tool = "poetry"
except Exception as exc:
    print(f"frozen: {type(exc).__name__}: {exc}")

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Ladder:
    """A named ladder of rungs, e.g. the packaging ladder from atlas page 8.

    Two things to fill in:
      1. `.climb_to(level)` -> the rungs up to AND INCLUDING that level, in order;
         an unknown level returns []
      2. `.top` -> the last rung, or None when the ladder is empty
    """

    name: str
    # Given, because it is the §4 trap in dataclass clothing: `= []` here would be
    # ONE list shared by every Ladder ever constructed. `field(default_factory=list)`
    # calls list() fresh per instance. Read it, then write the two methods.
    rungs: list[Rung] = field(default_factory=list)

    def climb_to(self, level: str) -> list[Rung]:
        raise NotImplementedError          # TODO 1

    @property
    def top(self) -> Rung | None:
        raise NotImplementedError          # TODO 2

In [ ]:
def _build():
    return Ladder("packaging", [Rung("S1", "uv venv"), Rung("S2", "uv build"), Rung("S3", "semver")])


check("Ladder defaults to no rungs", lambda: Ladder("empty").rungs, [])
check("climb_to is inclusive",
      lambda: [r.level for r in _build().climb_to("S2")], ["S1", "S2"])
check("climb_to on an unknown level is empty",
      lambda: _build().climb_to("S9"), [])
check("top is the last rung", lambda: _build().top.tool, "semver")
check("top of an empty ladder is None", lambda: Ladder("empty").top, None)

---
## 8 · Errors, exceptions, and exit codes

Atlas page 4's model: *exit codes and signals out*. Python's equivalent is the exception, and the
discipline is the same one from page 6 — fail loudly at the right altitude, and never swallow what
you cannot handle.

`except Exception:` around code you do not understand is how bugs become invisible. Catch the
narrowest type you can actually do something about, and use `raise ... from exc` to keep the chain.

In [ ]:
class ConfigError(Exception):
    """Domain error — a custom type lets callers catch precisely."""


def load_port(raw: str) -> int:
    try:
        port = int(raw)
    except ValueError as exc:
        raise ConfigError(f"port must be an integer, got {raw!r}") from exc   # ← chain it
    if not (1 <= port <= 65535):
        raise ConfigError(f"port {port} out of range")
    return port


for candidate in ("8000", "not-a-number", "99999"):
    try:
        print(f"{candidate!r} -> {load_port(candidate)}")
    except ConfigError as exc:
        print(f"{candidate!r} -> ConfigError: {exc}  (caused by {type(exc.__cause__).__name__})")

# else runs only when no exception; finally always runs
for value in ("1", "x"):
    try:
        n = int(value)
    except ValueError:
        print(f"{value!r}: failed")
    else:
        print(f"{value!r}: parsed {n}")
    finally:
        print(f"{value!r}: cleanup")

In [ ]:
class RetryExhausted(Exception):
    """Raised when every attempt failed. Must expose `.attempts` (int) and chain the last error."""


def retry(fn, attempts: int = 3):
    """Call fn(). On ANY exception, retry up to `attempts` times total.

    Return fn()'s value on the first success. If all attempts fail, raise
    RetryExhausted with .attempts set to the number tried, chained (`from`) to the
    last exception seen.

    This is atlas page 4's health-poll loop — retry, then report the failure honestly.
    """
    raise NotImplementedError

In [ ]:
def _flaky(fail_times):
    state = {"n": 0}

    def fn():
        state["n"] += 1
        if state["n"] <= fail_times:
            raise ValueError(f"attempt {state['n']} failed")
        return f"ok on attempt {state['n']}"
    return fn


def _drive_exhausted():
    try:
        retry(_flaky(99), attempts=3)
    except RetryExhausted as exc:
        return (exc.attempts, type(exc.__cause__).__name__)
    return "did not raise"


check("retry returns on first success", lambda: retry(_flaky(0)), "ok on attempt 1")
check("retry keeps trying", lambda: retry(_flaky(2), attempts=3), "ok on attempt 3")
check("retry reports attempts and chains the cause", _drive_exhausted, (3, "ValueError"))

---
## 9 · `pathlib` and file I/O

`pathlib.Path` replaces `os.path` string-mangling with objects. `/` is the join operator, which
reads far better than `os.path.join(a, b, c)`.

Atlas page 2's thesis is that everything on the desk is a door into one clone root — so most path
code you write is really *asking a question about that tree*. `Path` is the right tool for it.

In [ ]:
from pathlib import Path

home = Path.home()
clone_root = home / "src" / "github.com"        # `/` joins — no os.path.join needed

print(f"{clone_root}  exists={clone_root.exists()}  is_dir={clone_root.is_dir()}")

sample = Path("/opt/homebrew/bin/python3")
print(f"name={sample.name}  stem={sample.stem}  suffix={sample.suffix!r}  parent={sample.parent}")
print(f"parts={sample.parts}")

# round-trip through a temp file
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    note = Path(tmp) / "atlas" / "note.txt"
    note.parent.mkdir(parents=True, exist_ok=True)     # mkdir -p
    note.write_text("pipes make programs compose\n", encoding="utf-8")
    print(f"wrote {note.stat().st_size} bytes; read back: {note.read_text(encoding='utf-8')!r}")

# glob vs rglob: one level vs recursive
print(sorted(p.name for p in Path("/opt/homebrew/bin").glob("python3*"))[:5])

In [ ]:
from pathlib import Path


def largest_files(root: Path, suffix: str, top: int = 3) -> list[tuple[str, int]]:
    """Find the `top` largest files under `root` (recursively) whose suffix matches.

    Return [(filename, size_in_bytes), ...] sorted biggest-first. `suffix` includes
    the dot, e.g. ".py". Skip anything you cannot stat (broken symlinks, no permission)
    rather than crashing — atlas page 6: fail at the right altitude.
    """
    raise NotImplementedError


def depth_of(path: Path, root: Path) -> int:
    """How many directory levels `path` sits below `root`. depth_of(root, root) == 0.

    Hint: Path.relative_to, then count .parts.
    """
    raise NotImplementedError

In [ ]:
import tempfile
from pathlib import Path


def _fixture():
    tmp = Path(tempfile.mkdtemp())
    (tmp / "a").mkdir()
    (tmp / "a" / "big.py").write_text("x" * 300)
    (tmp / "a" / "small.py").write_text("x" * 100)
    (tmp / "mid.py").write_text("x" * 200)
    (tmp / "ignored.txt").write_text("x" * 9999)
    return tmp


_TMP = _fixture()

check("largest_files finds and ranks .py only",
      lambda: largest_files(_TMP, ".py"), [("big.py", 300), ("mid.py", 200), ("small.py", 100)])
check("largest_files honours top",
      lambda: largest_files(_TMP, ".py", top=1), [("big.py", 300)])
check("depth_of counts levels", lambda: depth_of(_TMP / "a" / "big.py", _TMP), 2)
check("depth_of root is zero", lambda: depth_of(_TMP, _TMP), 0)

---
## 10 · Modules, imports, and `__main__`

A module is a file; a package is a directory with importable contents. `sys.path` decides what is
findable — it is `$PATH` for imports, and it has the same *first match wins* rule you probed on
atlas page 3.

`if __name__ == "__main__":` is the guard that lets one file be both an importable library and a
runnable script. Without it, importing your script *runs* it.

In [ ]:
import sys
import tempfile
from pathlib import Path

print("sys.path, in resolution order:")
for i, entry in enumerate(sys.path[:6], 1):
    print(f"  {i}. {entry or '(cwd)'}")

# build a real module on disk and import it — same rule as $PATH: first match wins
scratch = Path(tempfile.mkdtemp())
(scratch / "atlas_demo.py").write_text(
    'VALUE = "imported"\n'
    'def shout(text):\n'
    '    return text.upper()\n'
    'if __name__ == "__main__":\n'
    '    print("run directly")\n',
    encoding="utf-8",
)

sys.path.insert(0, str(scratch))          # prepend → wins ties, like ~/.bun/bin on your PATH
import atlas_demo

print(f"\natlas_demo.VALUE = {atlas_demo.VALUE!r}")
print(f"atlas_demo.shout('pipes') = {atlas_demo.shout('pipes')!r}")
print(f"__name__ when imported  = {atlas_demo.__name__!r}   ← so the guard did NOT fire")
print(f"loaded from             = {atlas_demo.__file__}")

In [ ]:
import sys


def which_module(name: str) -> str | None:
    """Return the file a module WOULD load from, without importing it. None if not found.

    This is `which -a` for imports — atlas page 3's probe, in Python.
    Hint: importlib.util.find_spec.
    """
    raise NotImplementedError


def import_order_winner(name: str, candidate_dirs: list[str]) -> str | None:
    """Given candidate dirs, return the FIRST one that contains `name`.py, else None.

    Do not import anything and do not mutate sys.path — just answer the question.
    """
    raise NotImplementedError

In [ ]:
from pathlib import Path


check("which_module finds a stdlib module",
      lambda: Path(which_module("json")).name, "__init__.py")
check("which_module returns None for nonsense",
      lambda: which_module("definitely_not_a_real_module_xyz"), None)
check("import_order_winner picks the first hit",
      lambda: import_order_winner("atlas_demo", ["/nonexistent", str(scratch), "/tmp"]),
      str(scratch))
check("import_order_winner returns None when absent",
      lambda: import_order_winner("atlas_demo", ["/nonexistent", "/tmp"]), None)

---
## Scoreboard

Run this after working through the sections above. It reflects every check cell you have executed
in this kernel session — so restart-and-run-all for an honest number.

In [ ]:
scoreboard()

---
### Where to go next

Open **`02_atlas_probes.ipynb`** — same kernel, but instead of drilling syntax it turns each
atlas page's *"Probe the flow"* question into runnable Python against your actual machine.